In [1]:
# =========================================================
# 1. IMPORT LIBRARY
# =========================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [2]:
# =========================================================
# 2. MEMBUAT SPARK SESSION
# =========================================================

spark = SparkSession.builder \
    .appName("AirQualityClassification") \
    .getOrCreate()

print("Spark Session berhasil dibuat")

Spark Session berhasil dibuat


In [3]:
# =========================================================
# 3. MEMBACA DATASET
# =========================================================

df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("dataset/global air pollution dataset.csv")

print("Dataset berhasil dibaca")

Dataset berhasil dibaca


In [4]:
# =========================================================
# 4. RENAME KOLOM
# =========================================================

df = df.withColumnRenamed("AQI Value", "AQI_Value")
df = df.withColumnRenamed("AQI Category", "AQI_Category")

df = df.withColumnRenamed("CO AQI Value", "CO_AQI_Value")
df = df.withColumnRenamed("CO AQI Category", "CO_AQI_Category")

df = df.withColumnRenamed("Ozone AQI Value", "Ozone_AQI_Value")
df = df.withColumnRenamed("Ozone AQI Category", "Ozone_AQI_Category")

df = df.withColumnRenamed("NO2 AQI Value", "NO2_AQI_Value")
df = df.withColumnRenamed("NO2 AQI Category", "NO2_AQI_Category")

df = df.withColumnRenamed("PM2.5 AQI Value", "PM25_AQI_Value")
df = df.withColumnRenamed("PM2.5 AQI Category", "PM25_AQI_Category")

print(df.columns)

['Country', 'City', 'AQI_Value', 'AQI_Category', 'CO_AQI_Value', 'CO_AQI_Category', 'Ozone_AQI_Value', 'Ozone_AQI_Category', 'NO2_AQI_Value', 'NO2_AQI_Category', 'PM25_AQI_Value', 'PM25_AQI_Category']


In [5]:
# =========================================================
# 5. MENAMPILKAN DATA
# =========================================================

df.show(5, truncate=False)

+------------------+----------------+---------+------------+------------+---------------+---------------+------------------+-------------+----------------+--------------+-----------------+
|Country           |City            |AQI_Value|AQI_Category|CO_AQI_Value|CO_AQI_Category|Ozone_AQI_Value|Ozone_AQI_Category|NO2_AQI_Value|NO2_AQI_Category|PM25_AQI_Value|PM25_AQI_Category|
+------------------+----------------+---------+------------+------------+---------------+---------------+------------------+-------------+----------------+--------------+-----------------+
|Russian Federation|Praskoveya      |51       |Moderate    |1           |Good           |36             |Good              |0            |Good            |51            |Moderate         |
|Brazil            |Presidente Dutra|41       |Good        |1           |Good           |5              |Good              |1            |Good            |41            |Good             |
|Italy             |Priolo Gargallo |66       |Moderate

In [6]:
# =========================================================
# 6. EXPLORATORY DATA ANALYSIS
# =========================================================

print("Jumlah Data : ", df.count())

print("Jumlah Kolom : ", len(df.columns))

print("Nama Kolom : ")
for col_name in df.columns:
    print(col_name)

print("\nStruktur Dataset")
df.printSchema()

Jumlah Data :  23463
Jumlah Kolom :  12
Nama Kolom : 
Country
City
AQI_Value
AQI_Category
CO_AQI_Value
CO_AQI_Category
Ozone_AQI_Value
Ozone_AQI_Category
NO2_AQI_Value
NO2_AQI_Category
PM25_AQI_Value
PM25_AQI_Category

Struktur Dataset
root
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- AQI_Value: integer (nullable = true)
 |-- AQI_Category: string (nullable = true)
 |-- CO_AQI_Value: integer (nullable = true)
 |-- CO_AQI_Category: string (nullable = true)
 |-- Ozone_AQI_Value: integer (nullable = true)
 |-- Ozone_AQI_Category: string (nullable = true)
 |-- NO2_AQI_Value: integer (nullable = true)
 |-- NO2_AQI_Category: string (nullable = true)
 |-- PM25_AQI_Value: integer (nullable = true)
 |-- PM25_AQI_Category: string (nullable = true)



In [7]:
# =========================================================
# 7. STATISTIK DATASET
# =========================================================

df.describe().show()

+-------+-----------+----------+-----------------+--------------+------------------+--------------------+------------------+------------------+------------------+----------------+-----------------+-----------------+
|summary|    Country|      City|        AQI_Value|  AQI_Category|      CO_AQI_Value|     CO_AQI_Category|   Ozone_AQI_Value|Ozone_AQI_Category|     NO2_AQI_Value|NO2_AQI_Category|   PM25_AQI_Value|PM25_AQI_Category|
+-------+-----------+----------+-----------------+--------------+------------------+--------------------+------------------+------------------+------------------+----------------+-----------------+-----------------+
|  count|      23036|     23462|            23463|         23463|             23463|               23463|             23463|             23463|             23463|           23463|            23463|            23463|
|   mean|       NULL|       NaN|72.01086817542513|          NULL|1.3683672164684824|                NULL| 35.19370924434216|            

In [8]:
# =========================================================
# 8. CEK MISSING VALUE
# =========================================================

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+-------+----+---------+------------+------------+---------------+---------------+------------------+-------------+----------------+--------------+-----------------+
|Country|City|AQI_Value|AQI_Category|CO_AQI_Value|CO_AQI_Category|Ozone_AQI_Value|Ozone_AQI_Category|NO2_AQI_Value|NO2_AQI_Category|PM25_AQI_Value|PM25_AQI_Category|
+-------+----+---------+------------+------------+---------------+---------------+------------------+-------------+----------------+--------------+-----------------+
|    427|   1|        0|           0|           0|              0|              0|                 0|            0|               0|             0|                0|
+-------+----+---------+------------+------------+---------------+---------------+------------------+-------------+----------------+--------------+-----------------+



In [9]:
# =========================================================
# 9. DATA CLEANING
# =========================================================

df = df.dropna()

df = df.dropDuplicates()

print("Jumlah Data Setelah Cleaning : ", df.count())

Jumlah Data Setelah Cleaning :  23035


In [10]:
# =========================================================
# 10. DISTRIBUSI AQI CATEGORY
# =========================================================

df.groupBy("AQI_Category").count().show()

+--------------------+-----+
|        AQI_Category|count|
+--------------------+-----+
|           Unhealthy| 2215|
|                Good| 9688|
|Unhealthy for Sen...| 1568|
|           Hazardous|  191|
|            Moderate| 9087|
|      Very Unhealthy|  286|
+--------------------+-----+



In [11]:
# =========================================================
# 11. RATA-RATA AQI
# =========================================================

df.select(avg("AQI_Value")).show()

+-----------------+
|   avg(AQI_Value)|
+-----------------+
|72.34469285869329|
+-----------------+



In [12]:
# =========================================================
# 12. AQI TERTINGGI DAN TERENDAH
# =========================================================

print("AQI Tertinggi")
df.select(max("AQI_Value")).show()

print("AQI Terendah")
df.select(min("AQI_Value")).show()

AQI Tertinggi
+--------------+
|max(AQI_Value)|
+--------------+
|           500|
+--------------+

AQI Terendah
+--------------+
|min(AQI_Value)|
+--------------+
|             6|
+--------------+



In [13]:
# =========================================================
# 13. IMPORT MACHINE LEARNING
# =========================================================

from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [14]:
# =========================================================
# 14. FEATURE SELECTION
# =========================================================

df_ml = df.select(
    "AQI_Value",
    "CO_AQI_Value",
    "Ozone_AQI_Value",
    "NO2_AQI_Value",
    "PM25_AQI_Value",
    "AQI_Category"
)

df_ml.show(5)

+---------+------------+---------------+-------------+--------------+------------+
|AQI_Value|CO_AQI_Value|Ozone_AQI_Value|NO2_AQI_Value|PM25_AQI_Value|AQI_Category|
+---------+------------+---------------+-------------+--------------+------------+
|       64|           1|             15|            2|            64|    Moderate|
|       56|           1|             18|            6|            56|    Moderate|
|       91|           2|             22|            3|            91|    Moderate|
|       37|           1|             12|           13|            37|        Good|
|       88|           3|              7|            8|            88|    Moderate|
+---------+------------+---------------+-------------+--------------+------------+
only showing top 5 rows


In [15]:
# =========================================================
# 15. LABEL INDEXER
# =========================================================

labelIndexer = StringIndexer(
    inputCol="AQI_Category",
    outputCol="label"
)

In [16]:
# =========================================================
# 16. VECTOR ASSEMBLER
# =========================================================

assembler = VectorAssembler(
    inputCols=[
        "AQI_Value",
        "CO_AQI_Value",
        "Ozone_AQI_Value",
        "NO2_AQI_Value",
        "PM25_AQI_Value"
    ],
    outputCol="features"
)

In [17]:
# =========================================================
# 17. RANDOM FOREST
# =========================================================

rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=100,
    seed=42
)

In [18]:
# =========================================================
# 18. PIPELINE
# =========================================================

pipeline = Pipeline(
    stages=[
        labelIndexer,
        assembler,
        rf
    ]
)

In [19]:
# =========================================================
# 19. SPLIT DATA
# =========================================================

train_data, test_data = df_ml.randomSplit(
    [0.7, 0.3],
    seed=42
)

print("Training :", train_data.count())
print("Testing  :", test_data.count())

Training : 16198
Testing  : 6837


In [20]:
# =========================================================
# 20. TRAINING MODEL
# =========================================================

model = pipeline.fit(train_data)

print("Training selesai")

Training selesai


In [21]:
# =========================================================
# 21. PREDIKSI
# =========================================================

predictions = model.transform(test_data)

predictions.select(
    "AQI_Category",
    "prediction"
).show(20)

+------------+----------+
|AQI_Category|prediction|
+------------+----------+
|        Good|       0.0|
|        Good|       0.0|
|        Good|       0.0|
|        Good|       0.0|
|        Good|       0.0|
|        Good|       0.0|
|        Good|       0.0|
|        Good|       0.0|
|        Good|       0.0|
|        Good|       0.0|
|        Good|       0.0|
|        Good|       0.0|
|        Good|       0.0|
|        Good|       0.0|
|        Good|       0.0|
|        Good|       0.0|
|        Good|       0.0|
|        Good|       0.0|
|        Good|       0.0|
|        Good|       0.0|
+------------+----------+
only showing top 20 rows


In [22]:
# =========================================================
# 22. AKURASI
# =========================================================

evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)

print("Accuracy :", accuracy)

Accuracy : 0.9657744624835454


In [23]:
# =========================================================
# 23. PRECISION, RECALL, F1 SCORE
# =========================================================

precision = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

print("Precision :", precision.evaluate(predictions))
print("Recall    :", recall.evaluate(predictions))
print("F1 Score  :", f1.evaluate(predictions))

Precision : 0.9625771125737254
Recall    : 0.9657744624835454
F1 Score  : 0.9595160391623692


In [24]:
# =========================================================
# 24. CONFUSION MATRIX
# =========================================================

predictions.groupBy(
    "label",
    "prediction"
).count().show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  5.0|       2.0|   52|
|  5.0|       4.0|    4|
|  1.0|       1.0| 2715|
|  3.0|       2.0|   76|
|  4.0|       2.0|   78|
|  1.0|       0.0|    8|
|  2.0|       2.0|  636|
|  4.0|       4.0|   17|
|  0.0|       0.0| 2812|
|  1.0|       3.0|   16|
|  3.0|       3.0|  423|
+-----+----------+-----+



In [25]:
# =========================================================
# 25. STOP SPARK
# =========================================================

spark.stop()